# delucionqa Experiment Runner — OpenRouter variant

RAG ablation sweep for the **delucionqa** subset of `galileo-ai/ragbench`
(automotive owner's-manual QA), with all LLM calls through **OpenRouter**.
Run alongside the other OpenRouter notebooks — it uses its own
`config/`, `reports/`, `temp/`, and `cache_openrouter_delucionqa/` dirs.

Driven by `experiment_configs/delucionqa_openrouter_experiment.yaml`.
**Requires** `OPENROUTER_API_KEY` (comma-separate multiple keys to rotate).


## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas -q')



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [2]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
print('core.registry   loaded from:', _reg.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))


/Users/bhupendra.bhoi/pandas_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current directory: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support
experiment_runner loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/experiment/experiment_runner.py
core.registry   loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/core/registry.py
HuggingFace token loaded: True
Groq API key loaded: True
OpenRouter API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [14]:
EXPERIMENT_CONFIG_PATH = project_root / "experiment_configs/delucionqa_openrouter_validation.yaml"

experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Temp Dir:    {experiment_config.temp_dir}")
print(f"  Cache:       {experiment_config.cache}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  rag-experiments/delucionqa-openrouter-experiment/validation_config
  Report Dir:  rag-experiments/delucionqa-openrouter-experiment/reports
  Temp Dir:    rag-experiments/delucionqa-openrouter-experiment/temp
  Cache:       {'enabled': True, 'cache_dir': './cache_openrouter_delucionqa'}
  Num Queries: 100
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'delucionqa', 'split': 'test', 'limit': 184}

Data Parser:
  Type: noop


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [15]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


In [16]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/delucionqa (test)...
Loaded 184 samples

✅ Data loaded successfully!
  Documents: 235 parsed documents
  Raw Data:  184 samples

First Sample:
  Question: What if I fail to latch the tailgate properly?...
  Documents: 3


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [17]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f'Loaded {len(configs)} RAG pipeline configurations:')
for cfg in configs:
    searches = ' + '.join(s.type.value for s in cfg.retrieval.search.searches)
    fusion = cfg.retrieval.fusion.type.value if cfg.retrieval.fusion else '-'
    rerank = cfg.retrieval.rerank.type.value if cfg.retrieval.rerank else '-'
    qx = cfg.retrieval.query_transform.type.value if cfg.retrieval.query_transform else '-'
    gc = cfg.generation.config
    model = gc.get('model') if isinstance(gc, dict) else getattr(gc, 'model', None)
    print(f'  - {cfg.name}')
    print(f'      chunking={cfg.chunking.type.value}  embed={cfg.embedding.type.value}')
    print(f'      search=[{searches}]  fusion={fusion}  rerank={rerank}  q_transform={qx}')
    print(f'      generation_model={model}')


Loaded 2 RAG pipeline configurations:
  - delucionqa_or_v1_baseline
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - delucionqa_or_v3_embed_bge
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct


## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [18]:
get_ipython().system('pip install rank_bm25 -q')

# Run experiments


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [19]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 2 configurations from 0 to 100 queries...
Parallel mode: False
Progress: 20/100 (20.0%) | QPS: 204102.38 | ETA: 0s | Elapsed: 0sUsing key #2: ****fbdbUsing key #2: ****fbdb

HTTP 402 (retryable) on ****fbdb
Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}
{'provider': 'openrouter', 'current_index': 2, 'keys': [{'key_suffix': 'e916', 'available': True, 'cooldown_until': datetime.datetime(2026, 7, 25, 12, 44, 23, 259284), 'requests': 2, 'successes': 0, 'failures': 2, '429s': 2}, {'key_suffix': 'b076', 'available': True, 'cooldown_until': datetime.datetime(2026, 7, 25, 12, 44, 40, 636051), 'requests': 5, 'successes': 4, 'failures': 1, '429s': 1}, {'key_suffix': 'fbdb', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 25, 17, 44, 14, 811712), 'requests': 47, 'successes': 45, 'failu

2026-07-25 17:45:06,656 ERROR rag.pipeline.rag_pipeline: Query failed: No available OpenRouter API keys (all rate-limited/cooling down).


HTTP 402 (retryable) on ****fbdb
Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}
{'provider': 'openrouter', 'current_index': 2, 'keys': [{'key_suffix': 'e916', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 25, 17, 46, 5, 390389), 'requests': 3, 'successes': 0, 'failures': 3, '429s': 3}, {'key_suffix': 'b076', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 25, 17, 46, 5, 899037), 'requests': 6, 'successes': 4, 'failures': 2, '429s': 2}, {'key_suffix': 'fbdb', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 25, 17, 46, 6, 656553), 'requests': 48, 'successes': 45, 'failures': 3, '429s': 3}, {'key_suffix': '2462', 'available': False, 'cooldown_until': datetime.datetime(2026, 7, 25, 17, 46, 4, 451480), 'requests': 34, 'successes': 32, 'failures': 1, '4

2026-07-25 17:45:06,712 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,721 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,731 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,740 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,749 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,760 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,770 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:06,782 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45

Progress: 100/100 (100.0%) | QPS: 0.86 | ETA: 0s | Elapsed: 1.9m
Progress: 20/100 (20.0%) | QPS: 285326.80 | ETA: 0s | Elapsed: 0s

2026-07-25 17:45:10,564 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,565 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,591 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,591 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,619 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,619 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,649 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:10,649 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45

Progress: 71/100 (71.0%) | QPS: 70.80 | ETA: 0s | Elapsed: 1s

2026-07-25 17:45:11,527 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,528 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,556 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,556 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,592 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,592 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,625 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45:11,632 ERROR rag.pipeline.rag_pipeline: Query failed: All OpenRouter API keys are currently rate limited.
2026-07-25 17:45

Progress: 100/100 (100.0%) | QPS: 49.77 | ETA: 0s | Elapsed: 2s

✅ Experiments completed!
  Ran 2 configurations
  - delucionqa_or_v1_baseline: 100 queries
  - delucionqa_or_v3_embed_bge: 100 queries


## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [20]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

Record 21 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 2 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 22 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 23 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 24 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 25 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 26 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 27 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 28 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 29 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 30 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 31 evaluation failed: All OpenRouter API keys are currently rate limited.
Record 32 evaluation failed: 

Found 2 configs with JSONL files
[delucionqa_or_v1_baseline] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v1_baseline.jsonl
[delucionqa_or_v3_embed_bge] Evaluation complete → rag-experiments/delucionqa-openrouter-experiment/temp/delucionqa_or_v3_embed_bge.jsonl

✅ Evaluation complete: 2 configs


## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [21]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: rag-experiments/delucionqa-openrouter-experiment/reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [22]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: delucionqa_or_v1_baseline

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `delucionqa_or_v1_baseline`

**name**: delucionqa_or_v1_baseline  •  **mode**: test  •  **providers**: {'openrouter': {'type': 'openrouter', 'api_key_env': 'OPENROUTER_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'fixed_word', 'config': {'max_words': 160, 'overlap_words': 24}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 384}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 5}}]}, 'query_transform': None, 'fusion': None, 'rerank': None}  •  **generation**: {'strategy': 'default', 'provider': 'openrouter', 'config': {'model': 'meta-llama/llama-3.3-70b-instruct', 'temperature': 0.0, 'max_tokens': 512, 'system_prompt': 'You are an automotive owner\'s-manual support assistant. Answer questions\nabout vehicle features, controls, and procedures using ONLY the provided\nowner\'s-manual passages.\n\nCRITICAL RULES:\n1. Answer ONLY from the passages provided. Do not use outside knowledge.\n2. Give exact, step-by-step procedures. Preserve exact control names, button\n   names, warning/CAUTION text, and feature labels verbatim.\n3. Include any safety warnings (WARNING/CAUTION) that the passages attach to\n   the procedure.\n4. Be concise and procedural - give the steps, not background.\n5. If the passages do not contain the answer, respond with exactly:\n   "The passages do not provide sufficient information to answer this question."\n6. Every claim must be directly supported by the passages provided.\n', 'user_prompt': 'Passages:\n{context}\n\nQuestion: {query}\n\nAnswer (from passages only, give exact steps, control names, and any safety warnings):\n'}}  •  **evaluation**: {'type': 'trace', 'provider': 'openrouter', 'config': {'model': 'meta-llama/llama-3.3-70b-instruct', 'temperature': 0.0, 'max_tokens': 2000}}  •  **cache**: {'enabled': True, 'cache_dir': './cache_openrouter_delucionqa'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,What if I fail to latch the tailgate properly?,"[{'text': '', 'title': '', 'content': 'Closing To close the tailgate, lift u...",Failure to securely latch the tailgate could result in damage to the vehicle...,0.3333,0.1111,0.2222,0.2333,0.1111,0.1222,0.6000,1.0000,-0.4000,0.0,1.0,-1.0
1,What kind of safety features are implemented in this car?,"[{'text': '', 'title': '', 'content': 'SAFETY FEATURES When the Safety/Drivi...",The passages mention the following safety features:\n\n1. Occupant Restraint...,0.3077,0.2500,0.0577,0.5385,0.2500,0.2885,0.6250,1.0000,-0.3750,0.0,1.0,-1.0
2,When will the Automatic SOS be triggered?,"[{'text': '', 'title': '', 'content': 'Automatic SOS — If Equipped Automatic...",The passages do not provide sufficient information to answer this question.,0.0303,0.0244,0.0059,0.0000,0.0244,-0.0244,0.0000,1.0000,-1.0000,1.0,1.0,0.0
3,What happens if I accidentally push the SOS Call button?,"[{'text': '', 'title': '', 'content': 'Connected Services SOS FAQs — If Equi...","If you accidentally push the SOS Call button, you have 10 seconds to cancel ...",0.2857,0.0820,0.2037,0.2000,0.0656,0.1344,0.4000,0.8000,-0.4000,0.0,1.0,-1.0
4,What is the DEF?,"[{'text': '', 'title': '', 'content': 'Adding Diesel Exhaust Fluid The DEF g...","The passages do not provide a direct definition of DEF. However, according t...",0.7778,0.0625,0.7153,0.3889,0.0625,0.3264,0.5000,1.0000,-0.5000,0.0,1.0,-1.0
5,What may cause erratic or noisy performance of the radio?,"[{'text': '', 'title': '', 'content': 'Under certain conditions, the mobile ...",The mobile phone being on in your vehicle can cause erratic or noisy perform...,0.4211,0.1143,0.3068,0.4211,0.1143,0.3068,1.0000,1.0000,0.0000,1.0,1.0,0.0
6,how to calculate the gross trailer weight?,"[{'text': '', 'title': '', 'content': 'Gross Trailer Weight (GTW) The GTW is...","To calculate the Gross Trailer Weight (GTW), follow these steps:\n\n1. Put y...",0.1579,0.2727,-0.1148,0.2105,0.2727,-0.0622,1.0000,1.0000,0.0000,0.0,1.0,-1.0
7,What can the ASIST button do?,"[{'text': '', 'title': '', 'content': 'If equipped, the ASSIST Button is use...",The ASSIST button is used to automatically connect you to any one of the fol...,0.1522,0.4348,-0.2826,0.2174,0.4348,-0.2174,0.8571,1.0000,-0.1429,0.0,1.0,-1.0
8,What does the Door Off Mirror Kit do?,"[{'text': '', 'title': '', 'content': 'For information on the Door Off Mirro...",The Door Off Mirror Kit allows exterior rearview mirrors to be installed ont...,0.4400,0.1304,0.3096,0.4400,0.1304,0.3096,0.6364,1.0000,-0.3636,0.0,1.0,-1.0
9,Can I manually activate or deactivate the air conditioner?,"[{'text': '', 'title': '', 'content': 'The Air Conditioning (A/C) button all...","To manually activate or deactivate the air conditioner, press and release th...",0.2000,0.0811,0.1189,0.3429,0.1351,0.2078,0.8571,0.6667,0.1904,0.0,1.0,-1.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.3530,0.2060,0.2056,0.1651,0.2279
1,utilization_score,0.2953,0.1892,0.1731,0.1589,0.1781
2,completeness_score,0.6375,0.8762,0.3025,0.2242,0.3120
3,adherence_score,0.2500,1.0000,0.4330,0.0000,0.7500


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [23]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: rag-experiments/delucionqa-openrouter-experiment/reports/comparison.csv


In [24]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,delucionqa_or_v5_hybrid_rrf,0.1650,0.1236,0.1582,0.1069,0.7387,0.2826,0.1053,0.8947
1,delucionqa_or_v9_bge_sentence,0.2746,0.1700,0.2538,0.1555,0.6430,0.3026,0.2105,0.7895
2,delucionqa_or_v1_baseline,0.2979,0.1954,0.2374,0.1781,0.6330,0.3524,0.1579,0.8421
3,delucionqa_or_v3_embed_bge,0.3530,0.2279,0.2953,0.1781,0.6375,0.3120,0.2500,0.7500
4,delucionqa_or_v2_hybrid_wsum,0.1879,0.1545,0.1482,0.1172,0.6217,0.3709,0.1000,0.9000
5,delucionqa_or_v6_rerank_only,0.3494,0.1668,0.2861,0.1650,0.5997,0.3544,0.2000,0.8000
6,delucionqa_or_v7_hybrid_rerank,0.3639,0.1895,0.2622,0.1296,0.6291,0.2959,0.2500,0.7500
7,delucionqa_or_v4_chunk_sentence,0.4028,0.2419,0.2760,0.1647,0.5575,0.4034,0.2500,0.7500
8,delucionqa_or_v8_hyde,0.3707,0.1976,0.2692,0.1518,0.5349,0.3867,0.2500,0.7500


## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons